In [ ]:
import scanpy as sc
import scanpy
import os

sc.settings.verbosity = 3          # show progress
sc.logging.print_header()
scanpy.set_figure_params(dpi=100, facecolor='white')

# Output directory for figures
out_dir = "scRNA_seq_results"
os.makedirs(out_dir, exist_ok=True)

## Load data (public 10x Genomics dataset of 1,000 cells)
 ###    This is a small PBMC dataset but we will relabel as a placeholder.
 ###    For real ovarian data, replace URL with e.g. GEO accession.
 ###    Here we use a well-known 10x 3k PBMC dataset for demonstration.
 ###    To use actual ovarian cancer data, you can download from TISCH or GEO.

In [ ]:
print("Downloading example dataset (for demonstration).")
# Using 10x Genomics PBMC 3k dataset - works out of the box.
# For ovarian cancer, replace with e.g. GSEXXXXX_raw.h5ad
adata = sc.datasets.pbmc3k()   # 2,700 cells, ~13k genes

print(f"Dataset shape: {adata.shape[0]} cells, {adata.shape[1]} genes")

## Basic QC and Filtering

In [ ]:
# Mitochondrial genes (human: genes starting with MT-)
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# Filter cells: min 200 genes, max 5000 genes, max 20% mito
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_cells(adata, max_genes=5000)
adata = adata[adata.obs.pct_counts_mt < 20, :]

# Filter genes: expressed in at least 3 cells
sc.pp.filter_genes(adata, min_cells=3)

print(f"After QC: {adata.shape[0]} cells, {adata.shape[1]} genes")


## Normalisation & log transformation

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

## Highly variable genes (HVG) selection

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat', batch_key=None)
sc.pl.highly_variable_genes(adata, save=f"_{out_dir}/highly_variable_genes.png", show=False)
adata.raw = adata
adata = adata[:, adata.var.highly_variable]

# Regress out mitochondrial percentage and total counts (optional)
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])

# Scale data
sc.pp.scale(adata, max_value=10)

## Dimensionality reduction (PCA)

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True, save=f"_{out_dir}/pca_variance.png", show=False)

## Graph building and clustering (Leiden)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

## Visualise clusters

In [ ]:
sc.pl.umap(adata, color=['leiden', 'pct_counts_mt', 'total_counts'], 
           save=f"_{out_dir}/umap_clusters.png", show=False)

## Find marker genes per cluster

In [ ]:
import pandas as pd
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon', 
                         n_genes=20, use_raw=False)
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False, 
                        save=f"_{out_dir}/marker_genes.png", show=False)

# Save marker results to CSV
markers_df = pd.DataFrame(adata.uns['rank_genes_groups']['names']).head(10)
markers_df.to_csv(os.path.join(out_dir, "top_markers_per_cluster.csv"))

## Dot plot of known cancer-related TFs

In [ ]:
# Example TFs relevant to ovarian cancer (you can customise)
cancer_tfs = ['TP53', 'BRCA1', 'MYC', 'FOXM1', 'NFKB1', 'STAT3', 'JUN', 'FOS']
# Keep only TFs present in the dataset
present_tfs = [tf for tf in cancer_tfs if tf in adata.var_names]
if present_tfs:
    sc.pl.dotplot(adata, present_tfs, groupby='leiden', 
                  save=f"_{out_dir}/tf_dotplot.png", show=False)

## Save the annotated AnnData object for future use

In [ ]:
adata.write(os.path.join(out_dir, "ovarian_scRNAseq_processed.h5ad"))

print(f"\nAnalysis complete. Results saved in '{out_dir}' directory.")
print("Files generated:")
print("  - highly_variable_genes.png")
print("  - pca_variance.png")
print("  - umap_clusters.png")
print("  - marker_genes.png")
print("  - top_markers_per_cluster.csv")
print("  - tf_dotplot.png (if TFs found)")
print("  - ovarian_scRNAseq_processed.h5ad (full AnnData)")